# Game Translation A/B/C Experiment Runner

이 노트북은 아래 3조건을 **자동 실행**합니다.

- A: worldview prompt only
- B: glossary + translation rules
- C: glossary + translation rules + sentence type

실행 결과:
- `outputs/run_YYYY-MM-DD/A_outputs.csv`
- `outputs/run_YYYY-MM-DD/B_outputs.csv`
- `outputs/run_YYYY-MM-DD/C_outputs.csv`
- `eval/eval_sheet_prefilled_YYYY-MM-DD.csv`


In [ ]:
# If needed:
# %pip install -q openai pandas python-dotenv tqdm


In [ ]:
import os
import re
import time
from datetime import datetime
from pathlib import Path

import pandas as pd
from tqdm.auto import tqdm

try:
    from openai import OpenAI
except ImportError as e:
    raise ImportError("openai 패키지가 없습니다. 위 설치 셀을 먼저 실행하세요.") from e


In [ ]:
# =====================
# Config
# =====================
BASE_DIR = Path('/Users/yiji/Desktop/work/pseudo_lab/game_translation_exp')
DATA_DIR = BASE_DIR / 'data'
PROMPT_DIR = BASE_DIR / 'prompts'
EVAL_DIR = BASE_DIR / 'eval'

SAMPLES_PATH = DATA_DIR / 'samples.csv'  # sample_id,sentence_type,source_text
GLOSSARY_PATH = DATA_DIR / 'glossary.csv'
STYLE_GUIDE_PATH = DATA_DIR / 'style_guide.md'

WORLDVIEW_CONTEXT = """
Destiny-like sci-fantasy universe.
Tone: mysterious, heroic, high-stakes, mythic-technical blend.
UI text should be concise; gameplay mechanics must remain precise.
""".strip()

SRC_LANG = 'English'
TGT_LANG = 'Korean'

MODEL = os.getenv('OPENAI_MODEL', 'gpt-5.4-mini')
TEMPERATURE = 0.1
MAX_OUTPUT_TOKENS = 220

# DRY_RUN=True면 API 호출 없이 프롬프트만 점검합니다.
DRY_RUN = False

# API Key: 환경변수 OPENAI_API_KEY 필요
OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')
if not DRY_RUN and not OPENAI_API_KEY:
    raise ValueError('OPENAI_API_KEY 환경변수를 설정해주세요.')

RUN_DATE = datetime.now().strftime('%Y-%m-%d')
RUN_DIR = BASE_DIR / 'outputs' / f'run_{RUN_DATE}'
RUN_DIR.mkdir(parents=True, exist_ok=True)

print('RUN_DIR =', RUN_DIR)
print('MODEL =', MODEL)
print('DRY_RUN =', DRY_RUN)


In [ ]:
# Load templates
with open(PROMPT_DIR / 'A_worldview.txt', 'r', encoding='utf-8') as f:
    TEMPLATE_A = f.read()
with open(PROMPT_DIR / 'B_glossary_rules.txt', 'r', encoding='utf-8') as f:
    TEMPLATE_B = f.read()
with open(PROMPT_DIR / 'C_glossary_rules_type.txt', 'r', encoding='utf-8') as f:
    TEMPLATE_C = f.read()

# Load data
samples_df = pd.read_csv(SAMPLES_PATH)
glossary_df = pd.read_csv(GLOSSARY_PATH)
style_guide = STYLE_GUIDE_PATH.read_text(encoding='utf-8')

assert {'sample_id', 'sentence_type', 'source_text'}.issubset(samples_df.columns), 'samples.csv 컬럼 확인 필요'

print('samples:', len(samples_df))
print(samples_df['sentence_type'].value_counts(dropna=False))
print('glossary entries:', len(glossary_df))


In [ ]:
def glossary_to_text(df: pd.DataFrame) -> str:
    lines = []
    for _, r in df.iterrows():
        src = str(r.get('source_term', '')).strip()
        tgt = str(r.get('target_term', '')).strip()
        note = str(r.get('note', '')).strip()
        if src and tgt:
            if note and note.lower() != 'nan':
                lines.append(f'- {src} => {tgt} ({note})')
            else:
                lines.append(f'- {src} => {tgt}')
    return '\n'.join(lines)

GLOSSARY_TEXT = glossary_to_text(glossary_df)


def build_prompt(condition: str, source_text: str, sentence_type: str) -> str:
    common = {
        '{SRC_LANG}': SRC_LANG,
        '{TGT_LANG}': TGT_LANG,
        '{SOURCE_TEXT}': source_text,
        '{WORLDVIEW_CONTEXT}': WORLDVIEW_CONTEXT,
        '{GLOSSARY}': GLOSSARY_TEXT,
        '{STYLE_GUIDE_AND_RULES}': style_guide,
        '{SENTENCE_TYPE}': sentence_type,
    }

    if condition == 'A':
        prompt = TEMPLATE_A
    elif condition == 'B':
        prompt = TEMPLATE_B
    elif condition == 'C':
        prompt = TEMPLATE_C
    else:
        raise ValueError(f'Unknown condition: {condition}')

    for k, v in common.items():
        prompt = prompt.replace(k, v)
    return prompt


SYSTEM_MSG = 'You are a careful game localization assistant. Output translation only.'


def call_model(prompt: str) -> str:
    if DRY_RUN:
        return '[DRY_RUN]'

    client = OpenAI(api_key=OPENAI_API_KEY)

    resp = client.responses.create(
        model=MODEL,
        input=[
            {'role': 'system', 'content': SYSTEM_MSG},
            {'role': 'user', 'content': prompt},
        ],
        temperature=TEMPERATURE,
        max_output_tokens=MAX_OUTPUT_TOKENS,
    )

    text = (resp.output_text or '').strip()
    text = re.sub(r'\s+', ' ', text).strip()
    return text


In [ ]:
# Quick sanity check for one sample
ex = samples_df.iloc[0]
for cond in ['A', 'B', 'C']:
    p = build_prompt(cond, ex['source_text'], ex['sentence_type'])
    print(f'===== CONDITION {cond} PROMPT PREVIEW =====')
    print(p[:700] + ('...' if len(p) > 700 else ''))
    print()


In [ ]:
def run_condition(condition: str, df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for _, r in tqdm(df.iterrows(), total=len(df), desc=f'Run {condition}'):
        sid = r['sample_id']
        src = str(r['source_text'])
        stype = str(r.get('sentence_type', ''))

        prompt = build_prompt(condition, src, stype)
        out = call_model(prompt)

        rows.append({
            'sample_id': sid,
            'sentence_type': stype,
            'source_text': src,
            'translation': out,
        })

        if not DRY_RUN:
            time.sleep(0.15)

    return pd.DataFrame(rows)


A_df = run_condition('A', samples_df)
B_df = run_condition('B', samples_df)
C_df = run_condition('C', samples_df)

A_path = RUN_DIR / 'A_outputs.csv'
B_path = RUN_DIR / 'B_outputs.csv'
C_path = RUN_DIR / 'C_outputs.csv'

A_df[['sample_id', 'source_text', 'translation']].to_csv(A_path, index=False, encoding='utf-8')
B_df[['sample_id', 'source_text', 'translation']].to_csv(B_path, index=False, encoding='utf-8')
C_df[['sample_id', 'source_text', 'translation']].to_csv(C_path, index=False, encoding='utf-8')

print('Saved:')
print(A_path)
print(B_path)
print(C_path)


In [ ]:
# Build prefilled eval sheet
merged = samples_df[['sample_id', 'sentence_type', 'source_text']].copy()
merged = merged.merge(A_df[['sample_id', 'translation']].rename(columns={'translation': 'condition_A'}), on='sample_id', how='left')
merged = merged.merge(B_df[['sample_id', 'translation']].rename(columns={'translation': 'condition_B'}), on='sample_id', how='left')
merged = merged.merge(C_df[['sample_id', 'translation']].rename(columns={'translation': 'condition_C'}), on='sample_id', how='left')

for col in [
    'A_meaning','A_term','A_consistency','A_naturalness','A_ui_fit',
    'B_meaning','B_term','B_consistency','B_naturalness','B_ui_fit',
    'C_meaning','C_term','C_consistency','C_naturalness','C_ui_fit',
    'best','error_type','notes'
]:
    merged[col] = ''

eval_path = EVAL_DIR / f'eval_sheet_prefilled_{RUN_DATE}.csv'
merged.to_csv(eval_path, index=False, encoding='utf-8')
print('Saved:', eval_path)
print('Rows:', len(merged))


In [ ]:
# Quick scoring helper (optional)
# Fill scores first in eval_sheet_prefilled_YYYY-MM-DD.csv, then run this.

score_cols = [
    'A_meaning','A_term','A_consistency','A_naturalness','A_ui_fit',
    'B_meaning','B_term','B_consistency','B_naturalness','B_ui_fit',
    'C_meaning','C_term','C_consistency','C_naturalness','C_ui_fit',
]

ev = pd.read_csv(EVAL_DIR / f'eval_sheet_prefilled_{RUN_DATE}.csv')
for c in score_cols:
    ev[c] = pd.to_numeric(ev[c], errors='coerce')

summary = {
    'A_avg_total': ev[['A_meaning','A_term','A_consistency','A_naturalness','A_ui_fit']].mean(axis=1).mean(),
    'B_avg_total': ev[['B_meaning','B_term','B_consistency','B_naturalness','B_ui_fit']].mean(axis=1).mean(),
    'C_avg_total': ev[['C_meaning','C_term','C_consistency','C_naturalness','C_ui_fit']].mean(axis=1).mean(),
}
print(summary)
if 'best' in ev.columns:
    print('Best distribution:')
    print(ev['best'].value_counts(dropna=False))
